# Imports

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pulp as pl
from IPython.display import clear_output
import yfinance as yf
from scipy.stats import norm
from pulp import *

# Input Data

# VaR 

In [10]:
def VaR(ticker, p=0.95):
    
    # fetch all historical data for the ticker
    data = yf.download(ticker,start="1900-01-01", progress=False,auto_adjust=False)
    if data.empty:
        raise ValueError(f"No historical data found for ticker: {ticker}")

    returns = (data['Open']-data['Close'])/data['Open'] * -1
    return returns[returns.columns[0]].quantile(p)

# RoI

In [11]:
def RoI(ticker, p=0.95,days=365):
    import yfinance as yf
    from scipy.stats import norm

    # fetch all historical data for the ticker
    data = yf.download(ticker,start="1900-01-01" ,progress=False,auto_adjust=False)
    if data.empty:
        raise ValueError(f"No historical data found for ticker: {ticker}")

    returns = (data['Open']-data['Close'].shift(days))/data['Open']
    returns = returns[returns.columns[0]]
    E = returns.mean()
    return E 

# Optimization model

### pulp 2

In [ ]:
def optimize(tickers,budget,VaRs,RoIs,mxr,exr,ponder=0.5,diver=0.5):
    
    if mxr >= 1:
        mxr = mxr / 100
    if exr >= 1:
        exr = exr / 100

    # Penalty weights
    M1 = budget * ponder  # risk penalty
    M2 = budget * (1 - ponder)  # return penalty

    # Create the model
    model = LpProblem("Portfolio_Optimization", LpMaximize)

    # Decision variables
    stocks = {t: LpVariable(f"stocks_{t}", lowBound=0) for t in tickers}
    s1 = LpVariable("s1",lowBound=0)  # risk slack
    e1 = LpVariable("e1", lowBound=0)  # excess risk
    s2 = LpVariable("s2", lowBound=0)  # return slack
    e2 = LpVariable("e2", lowBound=0)  # excess return
    ts = LpVariable("ts", lowBound=0)  # total spent
    d = {t: LpVariable(f"d_{t}", lowBound=0, upBound=1,cat=LpBinary) for t in tickers}  # diversification binaries

    # Objective function: maximize return minus penalties
    model += (
        lpSum([stocks[t] * RoIs[t] for t in tickers])
        - lpSum([stocks[t] * VaRs[t] for t in tickers])
        - budget * s1 
        - budget * s2
        ,"Total_Return_Minus_Penalties"
    )


    # Constraints
    # Budget constraint
    model += lpSum([stocks[t] for t in tickers]) <= budget, "Max_Expenditure"

    # Total spent continuity
    model += lpSum([stocks[t] for t in tickers]) == ts, "Total_Spent"

    # Risk constraint (with slack)
    model += lpSum([stocks[t] * VaRs[t] for t in tickers]) - s1 + e1 == ts * mxr , "Risk_Constraint"

    # Return constraint (with slack)
    model += lpSum([stocks[t] * RoIs[t] for t in tickers]) + s2 - e2 == ts * exr , "Return_Constraint"

    # Diversification constraint
    model += lpSum([d[t] for t in tickers]) >= diver * len(tickers), "Diversification_Constraint"
    for t in tickers:
        model += stocks[t] <= d[t] * budget, f"Diversification_Link_{t}"

    # Solve
    model.solve()
    clear_output()

    class result:
        def __init__(self):
            pass
         
    r = result(); r.model = model; r.stocks = stocks; r.s1 = s1; r.s2 = s2; r.ts = ts

    return r

# Report

In [13]:
def df_to_md(mode,df,title):
    open("report.md",mode).write(f"\n\n# {title}\n\n")
    df.to_markdown("report.md",mode=mode)



In [14]:
budget = 100000
risk = 5
expected_return = 5
stocks = ["AAPL","^GSPC","MSFT","GOOG","AMZN","TSLA"]
days = 365
diver = 1
ponder= 0.5

In [15]:

VaRs = {ticker: VaR(ticker) for ticker in stocks}
RoIs = {ticker: RoI(ticker, days=days) for ticker in stocks}


z = optimize(tickers = stocks,
              budget=budget,
              VaRs=VaRs,
              RoIs=RoIs,
              mxr=risk,
              exr=expected_return,
              diver =diver,
              ponder = ponder
            )

varvals = {v.name: v.varValue for v in z.model.variables()}


# print inputs
inputs =pd.DataFrame({
    'Budget':"$"+str(budget),
    "Stocks":", ".join(stocks),
    "Risk (VaR)":str(risk)+"%",
    "Expected Return": str(expected_return)+"%",
    "Days": days,
},index=["Value"]).T
df_to_md(mode="w",df=inputs,title="Inputs")

# evironment
envs =pd.DataFrame({'VaR':VaRs,'RoI':RoIs},index=stocks)
df_to_md(mode="a",df=envs,title="Environment Parameters")


# variables
vars =pd.DataFrame([{v.name:v.varValue for v in z.model.variables()},
                    {v.name:v.dj for v in z.model.variables()}],
                    index=["Value","Reduced cost"]).T.map(lambda x: round(x,4))
df_to_md(mode="a",df=vars,title="Variables")


# constraints
constraints = pd.DataFrame([{c.name:c.pi for c in z.model.constraints.values()},
                    {c.name:c.slack for c in z.model.constraints.values()},
                    {c.name:c.value() for c in z.model.constraints.values()}],
                    index=["Dual Value","Slack","Value"]).T
df_to_md(mode="a",df=constraints,title="Constraints")

# solution statistics
stats =pd.DataFrame({
        "Risk": sum([varvals[f'stocks_{t}'] * VaRs[t] for t in stocks])/varvals['ts'],
        "Expected Return": sum([varvals[f'stocks_{t}'] * RoIs[t] for t in stocks])/varvals['ts'],
        "Pergcentage invested": z.ts.varValue/budget,
        "Number of stocks": sum([1 for t in stocks if varvals[f'stocks_{t}']> 0.0])
    },index=["Value"]).T
df_to_md(mode="a",df=stats,title="Solution Statistics")

reccomendations = []
# ponderation


# diversification analysis
if constraints[constraints.index.str.startswith("Diversification_Link")]["Dual Value"].sum() > 0:
    reccomendations.append("Consider increasing diversification to reduce risk.")
elif constraints[constraints.index.str.startswith("Diversification_Link")]["Dual Value"].sum() == 0:
    reccomendations.append("Diversification level is adequate.")
else:
    reccomendations.append("Consider decreasing diversification to increase returns.")


/tmp/ipykernel_34077/2401834832.py:49: RuntimeWarning: invalid value encountered in scalar divide
  "Risk": sum([varvals[f'stocks_{t}'] * VaRs[t] for t in stocks])/varvals['ts'],
/tmp/ipykernel_34077/2401834832.py:50: RuntimeWarning: invalid value encountered in scalar divide
  "Expected Return": sum([varvals[f'stocks_{t}'] * RoIs[t] for t in stocks])/varvals['ts'],


In [16]:
if stats.loc["Number of stocks","Value"] < len(stocks):
    if vars.loc["s1","Value"] < 0:
        print("You can increase the risk to potentially improve returns.")
    if vars.loc["s2","Value"] < 0:
        print("You can decrease the expected return to potentially reduce risk.")